In [1]:
import sys
sys.path.append("..")

In [5]:
from pathlib import Path

import numpy as np
from satpy import Scene

from src.preprocessing import AREA_EXTENT

CELL_SIZE = 5  # native pixels per cell side
THRESHOLDS = (0, 0.1, 0.5, 1, 2, 5)


def block_sum(field: np.ndarray, block_size: int) -> np.ndarray:
    """Sum non-overlapping blocks; rows and columns that do not fit are dropped."""
    n_rows = field.shape[0] // block_size * block_size
    n_cols = field.shape[1] // block_size * block_size
    trimmed_field = field[:n_rows, :n_cols]
    blocks = trimmed_field.reshape(
        n_rows // block_size, block_size, n_cols // block_size, block_size
    )
    return blocks.sum(axis=(1, 3))


li_files = sorted(Path("../data/raw/EO_EUM_DAT_0686").rglob("*CHK-BODY*.nc"))
all_cell_sums = []
for li_file in li_files:
    li_scene = Scene(filenames=[str(li_file)], reader="li_l2_nc")
    li_scene.load(["flash_accumulation"])
    italy_field = li_scene.crop(ll_bbox=AREA_EXTENT)["flash_accumulation"].values
    cell_sums = block_sum(np.nan_to_num(italy_field, nan=0.0), CELL_SIZE).ravel()
    all_cell_sums.append(cell_sums)
    print(li_scene.start_time, "| cells:", cell_sums.size, "| cells > 0:", int((cell_sums > 0).sum()))

all_cell_sums = np.concatenate(all_cell_sums)
for threshold in THRESHOLDS:
    print(f">= {threshold}: {float((all_cell_sums >= threshold).mean()):.4f}")

positive_cell_sums = all_cell_sums[all_cell_sums > 0]
print("percentiles of positive cells (50/75/90/99):", np.percentile(positive_cell_sums, [50, 75, 90, 99]).round(2))

2026-07-20 11:50:00 | cells: 8360 | cells > 0: 168
2026-07-20 12:00:00 | cells: 8360 | cells > 0: 165
2026-07-20 12:10:00 | cells: 8360 | cells > 0: 193
2026-07-20 12:20:00 | cells: 8360 | cells > 0: 218
2026-07-20 12:30:00 | cells: 8360 | cells > 0: 214
2026-07-20 12:40:00 | cells: 8360 | cells > 0: 208
2026-07-20 12:50:00 | cells: 8360 | cells > 0: 191
2026-07-20 13:00:00 | cells: 8360 | cells > 0: 228
>= 0: 1.0000
>= 0.1: 0.0204
>= 0.5: 0.0165
>= 1: 0.0143
>= 2: 0.0120
>= 5: 0.0086
percentiles of positive cells (50/75/90/99): [  2.14  11.99  63.16 380.64]


In [4]:
from pathlib import Path

from satpy import Scene

FCI_DIR = Path("../data/raw/EO_EUM_DAT_0662")
LI_DIR = Path("../data/raw/EO_EUM_DAT_0686")
CYCLE_TAG = "_N__O_0073_"  # repeat cycle 73 = 12:00-12:10

fci_files = sorted(str(p) for p in FCI_DIR.rglob("*CHK-BODY*.nc") if CYCLE_TAG in p.name)
li_files = sorted(str(p) for p in LI_DIR.rglob("*CHK-BODY*.nc") if CYCLE_TAG in p.name)
print(len(fci_files), "FCI files |", len(li_files), "LI files")  # expected: many | 1

fci_scene = Scene(filenames=fci_files, reader="fci_l1c_nc")
fci_scene.load(["ir_105"])
li_scene = Scene(filenames=li_files, reader="li_l2_nc")
li_scene.load(["flash_accumulation"])

fci_area = fci_scene["ir_105"].attrs["area"]
li_area = li_scene["flash_accumulation"].attrs["area"]

CENTRAL_COLUMN = 2784
for name, area in (("FCI", fci_area), ("LI", li_area)):
    latitude_row_1000 = area.get_lonlat(1000, CENTRAL_COLUMN)[1]
    latitude_row_4567 = area.get_lonlat(4567, CENTRAL_COLUMN)[1]
    print(f"{name}: row 1000 lat = {latitude_row_1000:.1f} | row 4567 lat = {latitude_row_4567:.1f}")
    
print("FCI shape:", fci_scene["ir_105"].shape, "| LI shape:", li_scene["flash_accumulation"].shape)
print("FCI extent:", fci_area.area_extent)
print("LI extent: ", li_area.area_extent)
print("same area:", fci_area == li_area)

40 FCI files | 1 LI files
FCI: row 1000 lat = -35.7 | row 4567 lat = 35.7
LI: row 1000 lat = 35.7 | row 4567 lat = -35.7
FCI shape: (5568, 5568) | LI shape: (5568, 5568)
FCI extent: (-5567999.994203024, 5567999.994203011, 5567999.994203011, -5567999.994203024)
LI extent:  (-5567999.994203018, -5567999.994203018, 5567999.994203017, 5567999.994203017)
same area: False
